# Statewide corn frames and block draws, 2025 (boundaries only)

States: PENNSYLVANIA, WISCONSIN. Upload `code_frames_a14bc9b.zip` and `nass_2025_state_targets.csv`, run all cells. Downloads the national CSB archive (3.76 GB) here, never on the laptop. Returns `frames_<fips>_2025.zip` per state (part field files, draw.json, state_acres.json) and its SHA256. Nothing is scored.

In [ ]:
# 1. Settings
import os, sys, json, hashlib, shutil, subprocess, zipfile
from pathlib import Path
UPLOAD, WORK, OUTD = Path("/content"), Path("/content/work"), Path("/content/out")
CODE = WORK / "code"
STATES = [["42", "PENNSYLVANIA"], ["55", "WISCONSIN"]]
YEAR, BLOCKS, PARTS, SEED, CRS = 2025, 150, 3, 42, "EPSG:5070"
CODE_ZIP, CODE_SHA, TARGETS = "code_frames_a14bc9b.zip", "6d4277e0ab33899159dab20ef308c79e3b7c7d2e35a04b5bdc649ef270ebf149", "nass_2025_state_targets.csv"
print("missing uploads:", [n for n in (CODE_ZIP, TARGETS) if not (UPLOAD / n).exists()] or "none")


In [ ]:
# 2. Optional: Google Drive keeps the full frames
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Installs (laptop versions, binary wheels), code check
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", *"numpy==2.5.4 pandas==3.1.1 pyarrow==26.0.1 shapely==2.2.0 pyproj==3.8.0 geopandas==1.2.0 pyogrio==0.13.0".split()], check=True)
assert hashlib.sha256((UPLOAD / CODE_ZIP).read_bytes()).hexdigest() == CODE_SHA, "code zip differs from the bundle"
shutil.rmtree(CODE, ignore_errors=True)
zipfile.ZipFile(UPLOAD / CODE_ZIP).extractall(CODE)
env = dict(os.environ, PYTHONPATH=str(CODE))
def run_py(*args):
    print("$", " ".join(map(str, args)), flush=True)
    r = subprocess.run([sys.executable, "-W", "ignore", *map(str, args)], env=env, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print(r.stderr[-3000:]); raise SystemExit("failed: " + str(args[0]))
OUTD.mkdir(parents=True, exist_ok=True)

In [ ]:
# 4. National CSB archive by range requests, then one subset per state
ZIP = WORK / "csb" / "NationalCSB_2018-2025_rev23.zip"
todo = [f for f, _ in STATES if not (OUTD / f"csb1825_{f}.parquet").exists()]
if todo:
    ZIP.parent.mkdir(parents=True, exist_ok=True)
    if not ZIP.exists() or ZIP.stat().st_size != 3758023527 or Path(str(ZIP) + ".done").exists():
        run_py(CODE / "scripts" / "range_download.py", "https://www.nass.usda.gov/sites/default/files/Research_and_Science/Crop-Sequence-Boundaries/datasets/NationalCSB_2018-2025_rev23.zip", ZIP, "--workers", "8")
    assert ZIP.stat().st_size == 3758023527
    for f in todo:
        run_py(CODE / "scripts" / "csb_subset.py", "--zip", ZIP, "--state", f, "--out", OUTD / f"csb1825_{f}.parquet")
    ZIP.unlink()

In [ ]:
# 5. Frame per state (corn + small-grain/corn double crops) and the seeded draw of read blocks
for f, name in STATES:
    frame_dir = OUTD / f"us_{f}_{YEAR}" / "frame"
    if not (frame_dir / "fields.geojson").exists():
        run_py("-m", "fieldvision.labels.from_file", OUTD / f"csb1825_{f}.parquet", "--where", f"STATEFIPS={f}",
               "--where", f"CDL{YEAR}=1,225,226,228,237", "--unlabelled", "--year", YEAR, "--id-column", "CSBID",
               "--no-year-in-id", "--prefix", "csb", "--country", "US", "--source", "nass-csb",
               "--crop-code-column", f"CDL{YEAR}", "--metric-crs", CRS,
               "--keep", "CSBACRES=acres_reported", "CNTYFIPS=cnty_fips", "CNTY=county_name", "--out", frame_dir)
    run_py(CODE / "scripts" / "state_sample.py", "--frame", frame_dir / "fields.geojson", "--state", f, "--year", YEAR,
           "--blocks", BLOCKS, "--parts", PARTS, "--seed", SEED, "--out", OUTD / f"us_{f}_{YEAR}",
           "--target-csv", UPLOAD / TARGETS, "--state-name", name)

In [ ]:
# 6. Small result zips (part field files + draw + state acres) and SHA256; full frames to Drive
from google.colab import files
for f, name in STATES:
    d = OUTD / f"us_{f}_{YEAR}"
    z_path = Path(f"/content/frames_{f}_{YEAR}.zip")
    with zipfile.ZipFile(z_path, "w", zipfile.ZIP_DEFLATED) as z:
        for k in range(1, PARTS + 1):
            z.write(d / f"s{k}" / "fields.geojson", f"s{k}/fields.geojson")
        for n in ("draw.json", "state_acres.json"):
            z.write(d / n, n)
        z.write(d / "frame" / "summary.json", "frame_summary.json")
        z.write(OUTD / f"csb1825_{f}.json", f"csb1825_{f}.json")
    s_path = Path(f"/content/frames_{f}_{YEAR}_SHA256SUMS.txt")
    s_path.write_text(f"{hashlib.sha256(z_path.read_bytes()).hexdigest()}  {z_path.name}\n")
    print(z_path.name, z_path.stat().st_size // 1024, "KB")
    if Path("/content/drive/MyDrive").exists():
        dd = Path("/content/drive/MyDrive/fieldvision_frames_" + str(YEAR)); dd.mkdir(parents=True, exist_ok=True)
        shutil.copy2(d / "frame_fields.geojson", dd / f"frame_{f}_{YEAR}.geojson")
        shutil.copy2(OUTD / f"csb1825_{f}.parquet", dd)
    files.download(str(s_path)); files.download(str(z_path))